In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from rapidfuzz import fuzz
from tqdm import tqdm
from pathlib import Path

In [2]:
print("=== AMAZON ML CHALLENGE 2026: TEST INFERENCE ===")

# 1. Load Test Candidates
print("1. Loading test candidates...")
candidates = pd.read_csv("../output/candidate_pairs_test.tsv", sep='\t')
candidates['candidate_entity_ids'] = candidates['candidate_entity_ids'].astype(str).str.split(',')
df_pairs = candidates.explode('candidate_entity_ids').replace('', np.nan).dropna()
df_pairs = df_pairs.rename(columns={'candidate_entity_ids': 'candidate_id'})

# 2. Load Raw Test Data
print("\n2. Loading raw test sources...")
s1 = pd.read_csv("../student_resource/dataset/test/test_source1.tsv", sep='\t') 
s2 = pd.read_csv("../student_resource/dataset/test/test_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/test/test_source3.tsv", sep='\t')

s2_s3_combined = pd.concat([s2, s3], ignore_index=True)
s1[['business_name', 'business_address']] = s1[['business_name', 'business_address']].fillna('')
s2_s3_combined[['business_name', 'business_address']] = s2_s3_combined[['business_name', 'business_address']].fillna('')

# 3. Merge Text Data
print("\n3. Merging text data onto test pairs...")
df_pairs = df_pairs.merge(
    s1[['entity_id', 'business_name', 'business_address']].rename(columns={
        'business_name': 's1_name', 'business_address': 's1_address'
    }),
    left_on='source1_entity_id', right_on='entity_id', how='left'
).drop(columns=['entity_id'])

df_pairs = df_pairs.merge(
    s2_s3_combined[['entity_id', 'business_name', 'business_address']].rename(columns={
        'business_name': 'cand_name', 'business_address': 'cand_address'
    }),
    left_on='candidate_id', right_on='entity_id', how='left'
).drop(columns=['entity_id'])

# 4. Compute Features
print("\n4. Computing similarity features...")
s1_names, cand_names = df_pairs['s1_name'].tolist(), df_pairs['cand_name'].tolist()
s1_addrs, cand_addrs = df_pairs['s1_address'].tolist(), df_pairs['cand_address'].tolist()

df_pairs['name_fuzz_ratio'] = [fuzz.ratio(s1, c) for s1, c in tqdm(zip(s1_names, cand_names), total=len(s1_names))]
df_pairs['name_token_sort'] = [fuzz.token_sort_ratio(s1, c) for s1, c in tqdm(zip(s1_names, cand_names), total=len(s1_names))]
df_pairs['addr_fuzz_ratio'] = [fuzz.ratio(s1, c) for s1, c in tqdm(zip(s1_addrs, cand_addrs), total=len(s1_addrs))]
df_pairs['addr_token_sort'] = [fuzz.token_sort_ratio(s1, c) for s1, c in tqdm(zip(s1_addrs, cand_addrs), total=len(s1_addrs))]

df_pairs['exact_name_match'] = (df_pairs['s1_name'].str.lower() == df_pairs['cand_name'].str.lower()).astype(np.uint8)
df_pairs['s1_missing_addr'] = (df_pairs['s1_address'] == '').astype(np.uint8)
df_pairs['cand_missing_addr'] = (df_pairs['cand_address'] == '').astype(np.uint8)
df_pairs['both_missing_addr'] = (df_pairs['s1_missing_addr'] & df_pairs['cand_missing_addr']).astype(np.uint8)

feature_cols = [
    'name_fuzz_ratio', 'name_token_sort', 'addr_fuzz_ratio', 'addr_token_sort',
    'exact_name_match', 's1_missing_addr', 'cand_missing_addr', 'both_missing_addr'
]
X_test = df_pairs[feature_cols]

# 5. Load Model & Predict
print("\n5. Running XGBoost Predictions...")
model = xgb.XGBClassifier()
model.load_model("../output/xgb_matching_model.json")

with open("../output/optimal_threshold.txt", "r") as f:
    best_threshold = float(f.read())
print(f"Applying threshold: {best_threshold}")

probs = model.predict_proba(X_test)[:, 1]
df_pairs['is_match'] = (probs >= best_threshold).astype(int)

# 6. Format Submission File
print("\n6. Building final submission file...")
# Keep only the positive matches
matches = df_pairs[df_pairs['is_match'] == 1]

# Group candidate IDs by source1_entity_id, separated by commas
submission_grouped = matches.groupby('source1_entity_id')['candidate_id'].apply(
    lambda x: ','.join(x)
).reset_index()
submission_grouped = submission_grouped.rename(columns={'candidate_id': 'matched_entity_ids'})

# Crucial: Amazon ML Challenge requires ALL Source 1 IDs to be in the submission file, even those with zero matches
final_submission = s1[['entity_id']].rename(columns={'entity_id': 'source1_entity_id'})
final_submission = final_submission.merge(submission_grouped, on='source1_entity_id', how='left').fillna("")

output_file = Path("../output/matching_results.tsv")
final_submission.to_csv(output_file, sep='\t', index=False)

print(f"\nSUCCESS! Final submission saved to {output_file}")
print(f"Total Source 1 entities processed: {len(final_submission):,}")
print(f"Entities that found at least one match: {len(submission_grouped):,}")

=== AMAZON ML CHALLENGE 2026: TEST INFERENCE ===
1. Loading test candidates...

2. Loading raw test sources...

3. Merging text data onto test pairs...

4. Computing similarity features...


100%|██████████| 51963564/51963564 [02:03<00:00, 419371.87it/s]



5. Running XGBoost Predictions...
Applying threshold: 0.8500000000000003

6. Building final submission file...

SUCCESS! Final submission saved to ..\output\matching_results.tsv
Total Source 1 entities processed: 1,732,544
Entities that found at least one match: 1,618,421
